In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
from etl.common import init_spark3

import re
import sys
import time
import importlib
import pandas as pd

BASE_DIR = Path("/apps/jupyter/users/nguyetnguyen/features")

sys.path[:0] = [str(path) for path in [
                    BASE_DIR / "common" / "src",
                    BASE_DIR / "device_refactored",
                ]
]

from common.agg_columns import (
    build_groupby_window_query,
    build_aggregate_columns,
    get_lxw_windows,
)
from src.device_helpers import (
    register_temp_view,
    build_ratio_column,
    build_days_since_column,
    build_tac_columns,
    build_datediff_column
)

import configs.configs as config

spark = init_spark3.setup(
    job_cfg={
        'executor.instances': 4,
        'executor.cores': 4,
        'executor.memory': '12g',
    },
    script_name="build_device_tac_behaviour_features"
)

In [ ]:
# ==============================================================================
# SOURCE
# ==============================================================================
def df_device_current_raw_weekly():
    return (
        spark.read.parquet(config.DEVICE_CURRENT_TAC_FEATURES_HC_SAMPLE_PATH)
        .filter(
            (F.col("date") > config.START_DATE_TAC_BEHAVIOUR_FEATURES)
            & (F.col("date") <= config.SNAPSHOT_DATE_STR)
        )
        .selectExpr("phone_number", "date", f"{config.DEVICE_CURRENT_TAC_COLUMN} as device_current_12w")
        .filter(F.col("phone_number").isNotNull() & F.col("device_current_12w").isNotNull())
    )

def df_pn_last_activated_date():
    return (
        spark.read.parquet(config.LAST_ACTIVATED_PATH)
        .filter(
            (F.col("date") > config.START_DATE_LAST_ACTIVATED_DATE)
            & (F.col("date") <= config.SNAPSHOT_DATE_STR)
        )
        .groupBy("phone_number")
        .agg(F.max("last_activated_date").alias("start_date"))
    )

def df_device_current_weekly():
    return (
        df_device_current_raw_weekly()
        .join(df_pn_last_activated_date(), on="phone_number", how="inner")
        .where("date >= start_date")
        .drop("start_date")
        .repartition(200, "phone_number")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )

# ==============================================================================
# BASE
# ==============================================================================
def df_device_base_weekly():
    w_order = Window.partitionBy("phone_number").orderBy("date")
    w_prev = w_order.rowsBetween(Window.unboundedPreceding, -1)
    w_latest = Window.partitionBy("phone_number").orderBy(F.col("date").desc())
    
    df = (
        df_device_current_weekly()
        .withColumn("current_device", F.first("device_current_12w", ignorenulls=True).over(w_latest))
        .withColumn("prev_device", F.lag("device_current_12w").over(w_order))
        .withColumn("is_switch", (
            F.col("device_current_12w").isNotNull()
            & F.col("prev_device").isNotNull()
            & (F.col("device_current_12w") != F.col("prev_device"))
        ).cast("int"))
        .withColumn("switch_group", F.sum("is_switch").over(w_order))
        .withColumn("week_diff", F.expr(f"datediff('{config.SNAPSHOT_DATE_STR}', date) / 7").cast("int"))
        .withColumn("streak_length", F.count("*").over(Window.partitionBy("phone_number", "switch_group")))
    )
    
    for w in config.WINDOW_WEEKS:
        df = df.withColumn(f"is_l{w}w", F.col("week_diff") < w)
        
    for w in config.SWITCH_GAP_WINDOWS_WEEKS:
        sw_col = f"switch_week_l{w}w"
        gap_col = f"switch_tac_gap_l{w}w"
        prev_col = f"prev_switch_week_l{w}w"
        window_start = F.expr(f"date_sub('{config.SNAPSHOT_DATE_STR}', {w * 7})")
        
        df = (
            df
            .withColumn(sw_col, F.when((F.col("is_switch") == 1) & F.col(f"is_l{w}w"), F.col("date")).otherwise(F.lit(None)))
            .withColumn(prev_col, F.last(sw_col, ignorenulls=True).over(w_prev))
            .withColumn(gap_col, F.when((F.col("is_switch") == 1) & F.col(f"is_l{w}w"), 
                F.datediff(F.col("date"), 
                           F.when(F.col(prev_col) < window_start, window_start)
                           .otherwise(F.col(prev_col))) / 7))
        )
        
    return df.persist(StorageLevel.MEMORY_AND_DISK)

def df_device_switch_features_lxw():
    base = df_device_base_weekly()
    source = register_temp_view(base, "df_device_base_weekly")
    
    stats = [
        ("sum(is_switch)", "device_switch_count"),
        ("avg(switch_tac_gap)", "device_switch_tac_gap_avg"),
        ("max(switch_tac_gap)", "device_switch_tac_gap_max"),
        ("min(switch_tac_gap)", "device_switch_tac_gap_min"),
    ]
    
    query, _ = build_groupby_window_query(
        agg_exprs=stats,
        windows=get_lxw_windows(config.WINDOW_WEEKS),
        snapshot_date=config.SNAPSHOT_DATE_STR,
        group_by=["phone_number"],
        source=source,
    )
    return spark.sql(query)

def df_device_streak_features_lxw():
    base = df_device_base_weekly()
    w_latest = Window.partitionBy("phone_number").orderBy(F.col("date").desc())
    
    df_latest = base.withColumn("rank", F.row_number().over(w_latest)).filter("rank = 1")
    
    source = register_temp_view(df_latest, "df_latest")
    prev_columns = ["streak_length"]
    columns, _ = build_aggregate_columns(prev_columns, ["identity"])
    
    query = f"select phone_number, {', '.join(columns)} from {source}"
    return spark.sql(query)

# ==============================================================================
# TAC BEHAVIOUR FEATURES
# ==============================================================================
def df_device_tac_behaviour_features_lxw():
    df = (
        df_device_switch_features_lxw()
        .join(df_device_streak_features_lxw(), on="phone_number", how="outer")
    )
    return df.select([
        F.when(F.isnan(c), None).otherwise(F.col(c)).alias(c)
        for c in df.columns
    ])

In [ ]:
for snapshot_date in pd.date_range('2023-12-31', '2024-10-31', freq='W-SUN'):
    start_time = time.time()
    snapshot_date_str = snapshot_date.strftime('%Y-%m-%d')
    
    config.SNAPSHOT_DATE_STR = snapshot_date_str
    config.SNAPSHOT_DATE = datetime.strptime(config.SNAPSHOT_DATE_STR, config.DATE_FORMAT)
    config.START_DATE_TAC_BEHAVIOUR_FEATURES = (config.SNAPSHOT_DATE - timedelta(days=104*7)).strftime(config.DATE_FORMAT)
    config.START_DATE_LAST_ACTIVATED_DATE = (config.SNAPSHOT_DATE - timedelta(days=2*7)).strftime(config.DATE_FORMAT)
    
    print(datetime.now(), config.SNAPSHOT_DATE_STR)
    
    df_device_tac_behaviour_features_lxw().write.mode('overwrite').parquet(f'{config.DEVICE_TAC_BEHAVIOUR_FEATURES_HC_SAMPLE_PATH}/date={config.SNAPSHOT_DATE_STR}')
    
    end_time = time.time()
    
    print(f'Done at: {datetime.now()} during {end_time - start_time}')
    print('-'*50)